In [ ]:
import torch
from sklearn.datasets import load_breast_cancer
from tabpfn_extensions import unsupervised
from tabpfn_extensions.unsupervised import experiments



# TabPFN 
### Idea 1: Apply balanced OCT on the full original dataset, then train a TFM per leaf (most similar to RF-TabPFN, creating subgroups of data)
### Compare to Idea 2: Just immediately train 1 TFM on the undersampled dataset

In [ ]:
from tabpfn import TabPFNClassifier
import numpy as np
import pandas as pd

def train_tabpfn_per_leaf_full(X_full_proc_df, y_full, leaf_assignments,
                               min_leaf_size=500):
    """
    Train a TabPFN model for each leaf using the *full original dataset*,
    not the undersampled training data.
    """

    unique_leaves = np.unique(leaf_assignments)
    leaf_models = {}

    print(f"OCT produced {len(unique_leaves)} leaves.")

    for leaf in unique_leaves:
        idx = np.where(leaf_assignments == leaf)[0]
        X_leaf = X_full_proc_df.iloc[idx]
        y_leaf = y_full.iloc[idx]

        print(f"Leaf {leaf}: {len(y_leaf)} samples")

        if (len(y_leaf) > 1000): # len(X_leaf) < min_leaf_size) | 
            print(f"  Skipping (too large).")
            continue
        print(f"Leaf {leaf}: {len(y_leaf)} samples")
        
        clf = TabPFNClassifier(
            device="cpu",
            n_estimators=1
        )
        clf.fit(X_leaf.values, y_leaf.values)
        leaf_models[leaf] = clf

        print(f"  Trained TabPFN for leaf {leaf}")

    return leaf_models
leaf_models = train_tabpfn_per_leaf_full(
    X_full_proc_df, 
    train_pd[target],
    leaf_assignments_full,
    min_leaf_size=500
)


In [ ]:

def assign_leaves_full_dataset(X_full, preprocessor, feature_names, oct_model):
    X_proc = preprocessor.transform(X_full)
    X_proc_df = pd.DataFrame(X_proc, columns=feature_names)
    leaf_assignments = oct_model.apply(X_proc_df)
    return X_proc_df, leaf_assignments
X_full_proc_df, leaf_assignments_full = assign_leaves_full_dataset(
    train_pd, preprocessor, feature_names, balanced_model
)
X_full_proc_df.head()

In [ ]:
import torch
from sklearn.datasets import load_breast_cancer
from tabpfn_extensions import unsupervised
from tabpfn_extensions.unsupervised import experiments

# Load data
df = load_breast_cancer(return_X_y=False)
X, y = df["data"], df["target"]
attribute_names = df["feature_names"]

# Initialize models
clf = TabPFNClassifier(n_estimators=4)
reg = TabPFNRegressor(n_estimators=4)
model_unsupervised = unsupervised.TabPFNUnsupervisedModel(
    tabpfn_clf=clf, tabpfn_reg=reg
)

# Run outlier detection
exp_outlier = unsupervised.experiments.OutlierDetectionUnsupervisedExperiment(
    task_type="unsupervised"
)
results = exp_outlier.run(
    tabpfn=model_unsupervised,
    X=torch.tensor(X, dtype=torch.float32),
    y=torch.tensor(y),
    attribute_names=attribute_names,
    indices=[4, 12],  # Analyze features 4 and 12
)

 Running double-facility sampling w=0.00
>>> Distance computation will use 42 features
5 Categorical features (one-hot encoded): ['INCOME_LEVEL', 'AGEGRP', 'SEX', 'REGION', 'lab_monitoring_intensity']
12 Numeric features (normalized): ['stage_2017', 'util_2017', '2017Q1_ckd_claims', '2017Q1_max_ckd_stage', '2017Q2_ckd_claims', '2017Q2_max_ckd_stage', '2017Q3_ckd_claims', '2017Q3_max_ckd_stage', '2017Q4_ckd_claims', '2017Q4_max_ckd_stage', 'total_lab_tests', 'nephrology_visit_count']
25 Binary features (unchanged): ['has_Hypertension', 'has_Type_2_Diabetes', 'has_Anemia', 'has_Hyperlipidemia', 'has_Acute_Kidney_Failure', 'has_Hyperparathyroidism', 'has_Kidney_Transplant', 'has_Vitamin_D_Deficiency', 'has_Long-term_Drug_Therapy', 'has_Hypothyroidism', 'has_Sleep_Apnea', 'Antihyperlipidemic Drugs, NEC (THRCLS_53)', 'Cardiac, Beta Blockers (THRCLS_51)', 'Cardiac, Calcium Channel (THRCLS_52)', 'Psychother, Antidepressants (THRCLS_69)', 'Cardiac Drugs, NEC (THRCLS_46)', 'Cardiac, ACE Inhibit

ValueError: force_direction must be one of the four options